# OCT Lecture Generation — Training Notebook

**Before running:**
1. Runtime → Change runtime type → GPU (T4)
2. Upload your `output/` folder (1,191 `*.training.jsonl` files) to Google Drive
   under `My Drive/chalk-talk/a4-train/output/`
3. Run all cells top to bottom

Checkpoints are saved to `My Drive/chalk-talk/a4-train/checkpoints/`

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/chalk-talk/a4-train'
DATA_DIR   = f'{DRIVE_ROOT}/output'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'

import os
os.makedirs(CKPT_DIR, exist_ok=True)

# Verify data is there
files = [f for f in os.listdir(DATA_DIR) if f.endswith('.training.jsonl')]
print(f'Found {len(files)} training files')

In [ ]:
# ── 2. Check GPU ─────────────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 3. Model ─────────────────────────────────────────────────────────────────
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

P_MID  = 0
P_UP   = 1
P_STOP = 2


class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=4096, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


def causal_mask(sz, device):
    return torch.triu(torch.ones(sz, sz, device=device, dtype=torch.bool), diagonal=1)


class StrokeDecoder(nn.Module):
    def __init__(self, d_ctx, d_model=128, n_layers=2, n_heads=4, max_len=200, dropout=0.3):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        self.p_embed  = nn.Embedding(3, 16)
        self.pt_proj  = nn.Linear(2 + 16, d_model)
        self.pos_emb  = nn.Embedding(max_len + 2, d_model)
        self.ctx_proj = nn.Linear(d_ctx, d_model)
        self.start_tok = nn.Parameter(torch.randn(d_model) * 0.02)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, activation='gelu', norm_first=True,
        )
        self.transformer = nn.TransformerDecoder(dec_layer, num_layers=n_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.xy_head = nn.Linear(d_model, 2)
        self.p_head  = nn.Linear(d_model, 3)

    def _encode_pts(self, xy, p, offset=1):
        T     = xy.size(1)
        p_emb = self.p_embed(p)
        enc   = self.pt_proj(torch.cat([xy, p_emb], -1))
        pos   = torch.arange(offset, offset + T, device=xy.device)
        return enc + self.pos_emb(pos).unsqueeze(0)

    def _start(self, B, device):
        return (self.start_tok.unsqueeze(0).unsqueeze(0).expand(B, 1, -1)
                + self.pos_emb(torch.zeros(1, dtype=torch.long, device=device)))

    def forward(self, ctx, tgt_xy, tgt_p, pad_mask=None):
        B, S, _ = tgt_xy.shape
        inp_pts = self._encode_pts(tgt_xy[:, :-1], tgt_p[:, :-1])
        inp     = torch.cat([self._start(B, ctx.device), inp_pts], dim=1)
        mem     = self.ctx_proj(ctx).unsqueeze(1)
        cmask   = causal_mask(S, ctx.device)
        out     = self.transformer(tgt=inp, memory=mem, tgt_mask=cmask,
                                   tgt_key_padding_mask=pad_mask)
        out     = self.norm(out)
        return self.xy_head(out), self.p_head(out)

    @torch.no_grad()
    def generate(self, ctx, max_len=None, temperature=1.0):
        max_len = max_len or self.max_len
        device  = ctx.device
        mem     = self.ctx_proj(ctx).unsqueeze(1)
        seq     = self._start(1, device)
        pts     = []
        for step in range(max_len):
            L     = seq.size(1)
            cmask = causal_mask(L, device)
            out   = self.norm(self.transformer(tgt=seq, memory=mem, tgt_mask=cmask))
            x     = float(self.xy_head(out[0,-1])[0].clamp(0,1))
            y     = float(self.xy_head(out[0,-1])[1].clamp(0,1))
            p     = int(torch.distributions.Categorical(
                        logits=self.p_head(out[0,-1]) / max(temperature,1e-6)).sample())
            pts.append([x, y, p])
            if p == P_STOP: break
            xy_t = torch.tensor([[[x,y]]], dtype=torch.float32, device=device)
            p_t  = torch.tensor([[p]], dtype=torch.long, device=device)
            new  = self._encode_pts(xy_t, p_t, offset=step+1)
            seq  = torch.cat([seq, new], dim=1)
            if seq.size(1) > max_len: seq = seq[:, -max_len:]
        return pts


class OCTModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=4, n_heads=8,
                 max_seq_len=2048, d_stroke=128, n_stroke_layers=2,
                 dropout=0.3, pad_idx=0):
        super().__init__()
        self.d_model    = d_model
        self.vocab_size = vocab_size
        self.pad_idx    = pad_idx
        self.word_embed  = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc     = SinusoidalPE(d_model, max_len=max_seq_len, dropout=dropout)
        self.style_embed = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, activation='gelu', norm_first=True,
        )
        self.transformer     = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm_out        = nn.LayerNorm(d_model)
        self.word_head       = nn.Linear(d_model, vocab_size, bias=False)
        self.word_head.weight = self.word_embed.weight
        self.stroke_decoder  = StrokeDecoder(
            d_ctx=d_model, d_model=d_stroke,
            n_layers=n_stroke_layers, n_heads=4, dropout=dropout,
        )
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.word_embed.weight, std=0.02)
        for name, p in self.named_parameters():
            if 'word_embed' in name or 'style_embed' in name: continue
            if p.dim() > 1 and 'weight' in name: nn.init.xavier_uniform_(p)
            elif 'bias' in name: nn.init.zeros_(p)

    def encode_words(self, word_ids, pad_mask=None):
        B, T  = word_ids.shape
        x     = self.word_embed(word_ids) + self.style_embed
        x     = self.pos_enc(x)
        cmask = causal_mask(T, word_ids.device)
        h     = self.transformer(x, mask=cmask, src_key_padding_mask=pad_mask, is_causal=True)
        h     = self.norm_out(h)
        return h, self.word_head(h)

    def decode_strokes(self, ctx, stroke_xy, stroke_p, stroke_pad=None):
        return self.stroke_decoder(ctx, stroke_xy, stroke_p, stroke_pad)

    def extract_ctx(self, hidden, batch_idx, word_pos):
        return hidden[batch_idx, word_pos]

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())

print('Model code loaded.')

In [ ]:
# ── 4. Data ──────────────────────────────────────────────────────────────────
import json, random
from collections import Counter
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from functools import partial

PAD, UNK, BOS, EOS, SILENT, PGBREAK = '<pad>', '<unk>', '<bos>', '<eos>', '<silent>', '<page_break>'
SPECIALS = [PAD, UNK, BOS, EOS, SILENT, PGBREAK]
MAX_STROKES_PER_WORD = 8
MAX_STROKE_SEQ       = 150
MAX_STROKE_CTX       = 256   # max stroke contexts per batch


def build_vocab(data_dir, min_freq=2):
    counter = Counter()
    for path in Path(data_dir).glob('*.training.jsonl'):
        for line in path.read_text().splitlines():
            if line.strip():
                tok = json.loads(line)
                if tok['type'] == 'word':
                    counter[tok['word']] += 1
    vocab = {s: i for i, s in enumerate(SPECIALS)}
    for word, freq in counter.most_common():
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    return vocab


def encode_strokes(strokes, max_total=MAX_STROKE_SEQ, max_per_word=MAX_STROKES_PER_WORD):
    xy_list, p_list = [], []
    for stroke in strokes[:max_per_word]:
        for i, pt in enumerate(stroke):
            xy_list.append([float(pt[0]), float(pt[1])])
            p_list.append(P_UP if i == len(stroke)-1 else P_MID)
            if len(xy_list) >= max_total - 1: break
        if len(xy_list) >= max_total - 1: break
    if not xy_list:
        return torch.zeros(0,2), torch.zeros(0, dtype=torch.long)
    xy_list.append([0.0, 0.0]); p_list.append(P_STOP)
    return torch.tensor(xy_list, dtype=torch.float32), torch.tensor(p_list, dtype=torch.long)


def augment_xy(xy):
    if xy.size(0) == 0: return xy
    scale = random.uniform(0.85, 1.15)
    return (xy * scale + torch.randn_like(xy) * 0.008).clamp(0.0, 1.0)


class OCTDataset(Dataset):
    def __init__(self, data_dir, vocab, augment=False, max_seq_len=2048,
                 unk_rate=0.05, min_words=5):
        self.vocab       = vocab
        self.augment     = augment
        self.max_seq_len = max_seq_len
        self.unk_rate    = unk_rate
        self.pad_id      = vocab[PAD]; self.unk_id = vocab[UNK]
        self.bos_id      = vocab[BOS]; self.eos_id = vocab[EOS]
        self.silent_id   = vocab[SILENT]; self.pgbrk_id = vocab[PGBREAK]
        self.episodes    = []
        self._load(data_dir, min_words)

    def _load(self, data_dir, min_words):
        paths = sorted(Path(data_dir).glob('*.training.jsonl'))
        for path in paths:
            lines  = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
            current = []
            for tok in lines:
                if tok['type'] == 'lesson_start':       current = [tok]
                elif tok['type'] in ('page_break','end'):
                    if current: self._process(current, min_words); current = []
                else:
                    if current: current.append(tok)
            if current: self._process(current, min_words)
        print(f'[dataset] {len(self.episodes)} episodes from {len(paths)} videos')

    def _process(self, tokens, min_words):
        meta   = tokens[0] if tokens[0]['type'] == 'lesson_start' else {}
        body   = tokens[1:] if tokens[0]['type'] == 'lesson_start' else tokens
        wids   = [self.bos_id]
        s_pos, s_xys, s_ps = [], [], []
        for tok in body:
            if len(wids) >= self.max_seq_len - 1: break
            t = tok['type']
            if   t == 'word':   wid = self.vocab.get(tok['word'], self.unk_id)
            elif t == 'silent': wid = self.silent_id
            else: continue
            pos = len(wids); wids.append(wid)
            for stroke_list in [tok.get('strokes', [])]:
                if stroke_list:
                    xy, p = encode_strokes(stroke_list)
                    if xy.size(0) > 1:
                        s_pos.append(pos); s_xys.append(xy); s_ps.append(p)
        wids.append(self.eos_id)
        if len(wids) - 2 < min_words: return
        self.episodes.append({
            'word_ids': wids, 'stroke_pos': s_pos,
            'stroke_xys': s_xys, 'stroke_ps': s_ps,
            'topic': meta.get('topic',''), 'tag': meta.get('tag',''),
        })

    def __len__(self):  return len(self.episodes)

    def __getitem__(self, idx):
        ep   = self.episodes[idx]
        wids = list(ep['word_ids'])
        if self.augment:
            wids = [self.unk_id if (w not in (self.bos_id,self.eos_id,self.silent_id,
                                              self.pgbrk_id,self.pad_id)
                                    and random.random() < self.unk_rate) else w
                    for w in wids]
        xys = [augment_xy(xy.clone()) if self.augment else xy.clone() for xy in ep['stroke_xys']]
        return {'word_ids': wids, 'stroke_pos': list(ep['stroke_pos']),
                'stroke_xys': xys, 'stroke_ps': [p.clone() for p in ep['stroke_ps']]}


def collate_fn(batch, pad_id=0):
    B     = len(batch)
    T_max = max(len(b['word_ids']) for b in batch)
    word_ids      = torch.full((B, T_max), pad_id, dtype=torch.long)
    word_pad_mask = torch.ones(B, T_max, dtype=torch.bool)
    for i, b in enumerate(batch):
        L = len(b['word_ids'])
        word_ids[i,:L] = torch.tensor(b['word_ids'], dtype=torch.long)
        word_pad_mask[i,:L] = False
    all_bidx, all_wpos, all_xy, all_p = [], [], [], []
    for i, b in enumerate(batch):
        for pos, xy, p in zip(b['stroke_pos'], b['stroke_xys'], b['stroke_ps']):
            all_bidx.append(i); all_wpos.append(pos)
            all_xy.append(xy);  all_p.append(p)
    if len(all_xy) > MAX_STROKE_CTX:
        idx      = random.sample(range(len(all_xy)), MAX_STROKE_CTX)
        all_bidx = [all_bidx[i] for i in idx]; all_wpos = [all_wpos[i] for i in idx]
        all_xy   = [all_xy[i]   for i in idx]; all_p    = [all_p[i]   for i in idx]
    strokes = None
    if all_xy:
        S_max    = max(xy.size(0) for xy in all_xy); N = len(all_xy)
        s_xy     = torch.zeros(N, S_max, 2)
        s_p      = torch.zeros(N, S_max, dtype=torch.long)
        s_pad    = torch.ones(N, S_max, dtype=torch.bool)
        for i,(xy,p) in enumerate(zip(all_xy,all_p)):
            S=xy.size(0); s_xy[i,:S]=xy; s_p[i,:S]=p; s_pad[i,:S]=False
        strokes = {'batch_idx': torch.tensor(all_bidx,dtype=torch.long),
                   'word_pos':  torch.tensor(all_wpos, dtype=torch.long),
                   'xy': s_xy, 'p': s_p, 'pad_mask': s_pad}
    return {'word_ids': word_ids, 'word_pad_mask': word_pad_mask, 'strokes': strokes}

print('Data code loaded.')

In [ ]:
# ── 5. Config ─────────────────────────────────────────────────────────────────
CFG = dict(
    d_model       = 256,
    n_layers      = 4,
    d_stroke      = 128,
    n_stroke_layers = 2,
    dropout       = 0.3,
    batch_size    = 32,
    lr            = 1e-3,
    lr_p3         = 2e-4,
    epochs_p1     = 20,
    epochs_p2     = 20,
    epochs_p3     = 30,
    val_frac      = 0.05,
    patience      = 5,
    min_freq      = 2,
    stroke_weight = 0.1,
)
print('Config:', CFG)

In [ ]:
# ── 6. Build vocab + dataset ──────────────────────────────────────────────────
vocab_path = f'{CKPT_DIR}/vocab.json'
if os.path.exists(vocab_path):
    vocab = json.loads(open(vocab_path).read())
    print(f'Loaded vocab: {len(vocab)} tokens')
else:
    vocab = build_vocab(DATA_DIR, min_freq=CFG['min_freq'])
    open(vocab_path,'w').write(json.dumps(vocab, ensure_ascii=False))
    print(f'Built vocab: {len(vocab)} tokens → {vocab_path}')

pad_id = vocab[PAD]

full_ds = OCTDataset(DATA_DIR, vocab, augment=False)
n_val   = max(1, int(len(full_ds) * CFG['val_frac']))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_ds.dataset.augment = True

coll     = partial(collate_fn, pad_id=pad_id)
train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                      collate_fn=coll, num_workers=2, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                      collate_fn=coll, num_workers=2)
print(f'Train: {n_train}  Val: {n_val}  Batches/epoch: {len(train_dl)}')

In [ ]:
# ── 7. Build model ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

model = OCTModel(
    vocab_size=len(vocab),
    d_model=CFG['d_model'], n_layers=CFG['n_layers'],
    d_stroke=CFG['d_stroke'], n_stroke_layers=CFG['n_stroke_layers'],
    dropout=CFG['dropout'], pad_idx=pad_id,
).to(device)
print(f'Model: {model.n_params/1e6:.2f}M params')

# Optionally resume from a saved checkpoint
# ckpt = torch.load(f'{CKPT_DIR}/phase1.best.pt', map_location=device)
# model.load_state_dict(ckpt['model_state'])
# print('Resumed from checkpoint')

In [ ]:
# ── 8. Training helpers ───────────────────────────────────────────────────────
import time
from tqdm.notebook import tqdm

STROKE_W = CFG['stroke_weight']

def word_loss(logits, word_ids, pad_id):
    return F.cross_entropy(logits[:,:-1].reshape(-1,logits.size(-1)),
                           word_ids[:,1:].reshape(-1), ignore_index=pad_id)

def stroke_loss(xy_pred, p_logits, xy_tgt, p_tgt, pad_mask):
    valid       = ~pad_mask
    is_not_stop = (p_tgt != P_STOP)
    xy_mask     = (valid & is_not_stop).float()
    l_xy = ((xy_pred - xy_tgt)**2).sum(-1)
    l_xy = (l_xy * xy_mask).sum() / xy_mask.sum().clamp_min(1.0)
    p_flat  = p_logits[valid]; t_flat = p_tgt[valid]
    l_p     = F.cross_entropy(p_flat, t_flat) if p_flat.size(0) > 0 \
              else p_logits.new_tensor(0.0)
    return l_xy, l_p

def set_phase(model, phase):
    for p in model.parameters(): p.requires_grad_(False)
    if phase == 1:
        for p in model.stroke_decoder.parameters(): p.requires_grad_(True)
        model.word_embed.weight.requires_grad_(True)
    elif phase == 2:
        for n,p in model.named_parameters():
            if 'stroke_decoder' not in n: p.requires_grad_(True)
    else:
        for p in model.parameters(): p.requires_grad_(True)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Phase {phase}: {trainable/1e6:.2f}M / {model.n_params/1e6:.2f}M trainable')

def run_batch(model, batch, phase, pad_id):
    wids = batch['word_ids'].to(device)
    wpad = batch['word_pad_mask'].to(device)
    s    = batch['strokes']
    hidden, logits = model.encode_words(wids, wpad)
    total  = wids.new_tensor(0.0, dtype=torch.float32)
    parts  = {}
    if phase in (2,3):
        lw = word_loss(logits, wids, pad_id)
        total = total + lw; parts['word'] = lw.item()
    if phase in (1,3) and s is not None:
        ctx = model.extract_ctx(hidden,
                                s['batch_idx'].to(device),
                                s['word_pos'].to(device))
        xy_pred, p_log = model.decode_strokes(
            ctx, s['xy'].to(device), s['p'].to(device), s['pad_mask'].to(device))
        lxy, lp = stroke_loss(xy_pred, p_log,
                              s['xy'].to(device), s['p'].to(device),
                              s['pad_mask'].to(device))
        ls = lxy + lp; total = total + STROKE_W * ls
        parts['s_xy'] = lxy.item(); parts['s_p'] = lp.item()
    parts['total'] = total.item()
    return total, parts

def run_phase(model, phase, epochs, lr, log_path, ckpt_path):
    set_phase(model, phase)
    opt   = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                               lr=lr, weight_decay=0.02)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.05)
    best_val  = float('inf'); no_improve = 0
    log_f = open(log_path, 'a')
    for epoch in range(1, epochs+1):
        t0 = time.time(); model.train()
        tr = {}; nb = 0
        for batch in tqdm(train_dl, desc=f'P{phase} ep{epoch}/{epochs}', leave=False):
            opt.zero_grad()
            total, parts = run_batch(model, batch, phase, pad_id)
            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            for k,v in parts.items(): tr[k] = tr.get(k,0.) + v
            nb += 1
        sched.step()
        model.eval(); vl = {}; nv = 0
        with torch.no_grad():
            for batch in val_dl:
                _, parts = run_batch(model, batch, phase, pad_id)
                for k,v in parts.items(): vl[k] = vl.get(k,0.) + v
                nv += 1
        ta = {k:v/max(nb,1) for k,v in tr.items()}
        va = {k:v/max(nv,1) for k,v in vl.items()}
        vl_total = va.get('total', float('inf'))
        rec = {'phase':phase,'epoch':epoch,'train':ta,'val':va,
               'elapsed':round(time.time()-t0,1)}
        log_f.write(json.dumps(rec)+'\n'); log_f.flush()
        print(f'  ep{epoch:03d}  train={ta.get("total",0):.4f}  '
              f'val={vl_total:.4f}  ({time.time()-t0:.0f}s)')
        if vl_total < best_val:
            best_val = vl_total; no_improve = 0
            torch.save({'phase':phase,'epoch':epoch,
                        'model_state':model.state_dict(),'val_loss':vl_total},
                       ckpt_path)
            print(f'    ✓ saved  (val={vl_total:.4f})')
        else:
            no_improve += 1
            if no_improve >= CFG['patience']:
                print(f'    early stop'); break
    log_f.close()
    print(f'Phase {phase} done. Best val: {best_val:.4f}')

print('Training helpers loaded.')

In [ ]:
# ── 9. Phase 1: stroke decoder ───────────────────────────────────────────────
print('=== Phase 1: Stroke decoder only ===')
run_phase(model, phase=1, epochs=CFG['epochs_p1'], lr=CFG['lr'],
          log_path=f'{CKPT_DIR}/phase1.log.jsonl',
          ckpt_path=f'{CKPT_DIR}/phase1.best.pt')

In [ ]:
# ── 10. Phase 2: word LM ─────────────────────────────────────────────────────
print('=== Phase 2: Word LM only ===')
run_phase(model, phase=2, epochs=CFG['epochs_p2'], lr=CFG['lr'],
          log_path=f'{CKPT_DIR}/phase2.log.jsonl',
          ckpt_path=f'{CKPT_DIR}/phase2.best.pt')

In [ ]:
# ── 11. Phase 3: joint ───────────────────────────────────────────────────────
print('=== Phase 3: Joint training ===')
run_phase(model, phase=3, epochs=CFG['epochs_p3'], lr=CFG['lr_p3'],
          log_path=f'{CKPT_DIR}/phase3.log.jsonl',
          ckpt_path=f'{CKPT_DIR}/phase3.best.pt')
print('\nAll phases complete!')

In [ ]:
# ── 12. Quick generation test ─────────────────────────────────────────────────
# Load best phase 3 checkpoint and generate one word's strokes
ckpt = torch.load(f'{CKPT_DIR}/phase3.best.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Generate strokes for a sample word (index of 'the')
word = 'the'
wid  = vocab.get(word, vocab[UNK])
with torch.no_grad():
    wids   = torch.tensor([[vocab[BOS], wid]], dtype=torch.long, device=device)
    hidden, _ = model.encode_words(wids)
    ctx    = hidden[0, 1:2]   # context at position of 'the'
    pts    = model.stroke_decoder.generate(ctx, temperature=0.8)

print(f'Generated {len(pts)} points for "{word}"')
print('First 5 points:', pts[:5])

# Quick visualization
import matplotlib.pyplot as plt
strokes = []
cur = []
for x, y, p in pts:
    cur.append((x, y))
    if p >= P_UP:
        strokes.append(cur); cur = []
if cur: strokes.append(cur)

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
for s in strokes:
    xs, ys = zip(*s) if s else ([], [])
    ax.plot(xs, [1-y for y in ys], 'b-', linewidth=1.5)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title(f'Generated strokes for "{word}"')
ax.axis('off')
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/sample_{word}.png', dpi=120)
plt.show()
print(f'Saved to {CKPT_DIR}/sample_{word}.png')